# L3c Example: Recursive Implementation of Fibonacci Sequence Calculation
In this example, we illustrate recursion concepts by benchmarking three implementations of the Fibonacci sequence computation using [the BenchmarkTools.jl package](https://github.com/JuliaCI/BenchmarkTools.jl).

> __Learning Objectives:__
>
> By the end of this example, students will be able to:
> * __Measure the runtime performance of Fibonacci functions:__ Measure the runtimes of the loop-based, recursive, and memoized Fibonacci functions at the same input value. Verify that each function computes the correct Fibonacci values before comparing its runtime with the other functions.
> * __Analyze repeated work in a recursive function:__ Trace the calls made by the recursive Fibonacci function and identify subproblems that it computes more than once. Explain how these repeated calculations affect its runtime relative to the loop-based function.
> * __Explain and benchmark memoization:__ Explain how checking for a stored result before making another recursive call prevents repeated calculations. Configure the benchmark so that each sample begins with an empty dictionary and uses the same input value as the other implementations.

This example checks the three implementations for correctness and measures their runtimes at the same input value.
___

## Setup, Data, and Prerequisites
First, we set up the computational environment by including the `Include.jl` file and loading the packages and files we need.

> The [`include(...)` command](https://docs.julialang.org/en/v1/base/base/#include) evaluates the contents of the input source file, `Include.jl`, in the notebook's global scope. The `Include.jl` file sets paths, loads required external packages, etc. For additional information on functions and types used in this material, see the [Julia programming language documentation](https://docs.julialang.org/en/v1/).

Let's set up our code environment:

In [ ]:
include(joinpath(@__DIR__, "Include.jl")); # include the Include.jl file

The three Fibonacci implementations we benchmark come from [the `VLDataScienceMachineLearningPackage.jl` package](https://github.com/varnerlab/VLDataScienceMachineLearningPackage.jl); see [the documentation](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/). This example also uses `Base`, [the `Test` standard library](https://docs.julialang.org/en/v1/stdlib/Test/), and [the BenchmarkTools.jl package](https://github.com/JuliaCI/BenchmarkTools.jl).

### Implementations
The three versions we benchmark are in [the `Recursion.jl` file](../../../code/src/Recursion.jl) of the course package, which [`Include.jl`](Include.jl) loads for us.

* __Vanilla loop-based implementation:__ The `fibonacci(n::Int64)::Dict{Int64, Int64}` function fills a dictionary from `0` to `n` with a single for-loop.
* __Standard recursive implementation:__ The `fibonacci!(n::Int64, series::Dict{Int64, Int64})::Int64` function computes $F_{n}$ straight from the recurrence. It stores each index it visits in `series`, but it does not check `series` before recursing, so the same subproblems are recomputed many times.
* __Memoized recursive implementation:__ The `memoization_fibonacci!(n::Int64, series::Dict{Int64, Int64})::Int64` function is the same recursion with one extra line: it returns a stored value when it finds one, so each subproblem is computed exactly once.

The loop implementation returns the whole dictionary. Both recursive implementations write every index they visit into the `series` dictionary they are handed and return $F_{n}$ as an `Int64`.


### Constants
Let's set the constants this example uses. The `correct_fibonacci_sequence` constant holds the values we check each implementation against, and `benchmark_index` fixes the problem size, which every case uses so the three timings can be compared.

In [ ]:
correct_fibonacci_sequence = Dict(0 => 0, 1 => 1, 2 => 1, 3 => 2, 4 => 3, 5 => 5, 6 => 8, 7 => 13,
                                  8 => 21, 9 => 34, 10 => 55, 11 => 89, 12 => 144, 13 => 233,
                                  14 => 377, 15 => 610); # F0 through F15, so 16 known values
benchmark_index = 25; # every case below is benchmarked at this same n, so the timings are comparable

___

## Case 1: Test the for loop implementation of Fibonacci computation
In this case, we test and then time the for-loop implementation of the Fibonacci calculation. This is the baseline for the two recursive versions, since a runtime only means something next to another runtime.

Let's use the [BenchmarkTools.jl package](https://github.com/JuliaCI/BenchmarkTools.jl) to compute the average time required to calculate the sequence $F_{0},\dots,F_{n}$ using the vanilla implementation of the `fibonacci` function (for-loop-based implementation). However, before we benchmark the for loop implementation, let's check that it is correct by [using the `@test` macro exported by the `Test.jl` package](https://docs.julialang.org/en/v1/stdlib/Test/).

In [ ]:
let

    # initialize -
    number_of_test_terms = 15; # the reference dictionary holds F0 through F15
    my_computed_sequence = fibonacci(number_of_test_terms);

    for i ∈ 0:number_of_test_terms
        @test my_computed_sequence[i] == correct_fibonacci_sequence[i];
    end
end

Now that we have verified correctness, let's benchmark the for-loop implementation of the Fibonacci sequence calculation.

> __How we time each version:__ The [BenchmarkTools.jl package](https://github.com/JuliaCI/BenchmarkTools.jl) exports the [@benchmarkable macro](https://juliaci.github.io/BenchmarkTools.jl/stable/reference/#BenchmarkTools.@benchmarkable-Tuple), which builds a benchmark for a call without running it. Running that benchmark executes the call many times and returns a trial holding the runtime and memory profile of the run. We fix `samples` and `evals` explicitly rather than calling `tune!`, so every case runs at the same sample count and the same problem size. The reason `evals = 1` matters shows up in Case 3.

One difference stays. This case builds its own dictionary inside the timed call, where Cases 2 and 3 are handed an empty one before the clock starts, so read the runtimes against each other but expect the allocation counts to differ by that dictionary. The `result_basal::BenchmarkTools.Trial` variable holds the timing and allocation results of the run:

In [ ]:
result_basal = let
    test_run_basal = @benchmarkable fibonacci($(benchmark_index));
    result_basal = run(test_run_basal; samples = 200, evals = 1)
end

___

## Case 2: Test the recursive implementation of the Fibonacci computation
In this case, we test and then time the plain recursive implementation at the same problem size used in Case 1. The recursion states the Fibonacci rule almost exactly as it is written in mathematics, which makes its runtime surprising.

Next, let's benchmark a recursive implementation. The `fibonacci!(n::Int64, series::Dict{Int64, Int64})::Int64` function is a mutating recursive function that computes the sequence $F_{0},\dots, F_{n}$ for a given $n$. The recursive sequence is stored in the `series::Dict{Int64, Int64}` argument. This takes advantage of [the mutating function behavior](https://docs.julialang.org/en/v1/manual/functions/#man-argument-passing) in Julia, which allows us to update the dictionary in place without returning a new dictionary.

Let's verify that the recursive implementation is correct by checking that it computes the Fibonacci sequence correctly for $F_{0},\dots,F_{n}$, where $n$ is the largest index in the `correct_fibonacci_sequence` dictionary.

In [ ]:
let

    # initialize -
    number_of_test_terms = 15; # the reference dictionary holds F0 through F15
    my_computed_sequence = Dict{Int64, Int64}(); # initialize an empty dictionary
    fibonacci!(number_of_test_terms, my_computed_sequence); # mutates the dictionary in place; the returned Fn is ignored here

    # verify correctness - for terms 0 ... number_of_test_terms
    for i ∈ 0:number_of_test_terms
        @test my_computed_sequence[i] == correct_fibonacci_sequence[i];
    end
end

The tests pass, so the recursion computes the right values. How does it perform relative to the loop? The `result_recursive::BenchmarkTools.Trial` variable holds that measurement, taken at the same problem size and sample count as Case 1:

In [ ]:
result_recursive = let
    test_run_recursive = @benchmarkable fibonacci!($(benchmark_index), series) setup=(series = Dict{Int,Int}())
    result_recursive = run(test_run_recursive; samples = 200, evals = 1)
end

At the same `benchmark_index`, the plain recursive function is much slower than the loop. Each call for $F_n$ makes additional calls for $F_{n-1}$ and $F_{n-2}$ without checking whether those values were already computed. As a result, the same subproblems appear in many branches of the call tree, and the total number of calls grows exponentially with `n`. 

<div>
    <center>
      <img
        src="figs/Fig-Fibonacci-Recursive.svg"
        alt="Call tree for the recursive Fibonacci computation, showing repeated subtrees"
        height="400"
        width="800"
      />
    </center>
  </div>

The slowdown comes from this growing amount of repeated work, not from recursion alone.

___

## Case 3: Test the recursive implementation of the Fibonacci computation with memoization
In this case, we test and then time the same recursion again, with memoization added so each subproblem is computed only once. Case 2 measured the cost when the repeated work is present, and this case measures what we get back by removing it.

Finally, let's benchmark a recursive Fibonacci function that uses memoization. The `memoization_fibonacci!(n::Int64, series::Dict{Int64, Int64})::Int64` implementation is a mutating recursive function that uses memoization to speed up the computation of the sequence $F_{0},\dots, F_{n}$ for a given $n$. The recursive sequence is stored in the `series::Dict{Int64, Int64}` argument.

> __Why is there no memory cost here?__ Memoization normally trades memory for time: you keep a table of answers you would otherwise recompute. This example does not show that trade. Our plain recursion was already handed a dictionary to record every index it visits, so both recursive versions finish with the same `series` holding $F_{0},\dots,F_{n}$, and the memoized version adds one lookup rather than one table.

First, does this implementation do what we expect? Let's verify that it computes the Fibonacci sequence correctly for $F_{0},\dots,F_{n}$, where $n$ is the largest index in the `correct_fibonacci_sequence` dictionary.

In [ ]:
let

    # initialize -
    number_of_test_terms = 15; # the reference dictionary holds F0 through F15
    my_computed_sequence = Dict{Int64, Int64}(); # initialize an empty dictionary
    memoization_fibonacci!(number_of_test_terms, my_computed_sequence); # mutates the dictionary in place; the returned Fn is ignored here

    # verify correctness - for terms 0 ... number_of_test_terms
    for i ∈ 0:number_of_test_terms
        @test my_computed_sequence[i] == correct_fibonacci_sequence[i];
    end
end

Does memoization change the runtime and allocation profile of the recursive implementation? Let's benchmark the memoized recursion at the same problem size and sample count as Case 2. The `result_recursive_memo::BenchmarkTools.Trial` variable holds that measurement.

> __Why both benchmark settings matter:__
>
> An __evaluation__ is one timed call to the function. A __sample__ is a timing measurement that can combine one or more evaluations; here, the benchmark collects up to 200 samples.
>
> Before each sample, `setup=` creates a new empty dictionary outside the timed code. This prevents one sample from using Fibonacci values stored by an earlier sample.
>
> Setting `evals = 1` also gives each fresh dictionary to exactly one timed call. If `evals = 5`, the first call would fill the dictionary and the next four would reuse it, so the five calls would not measure the same work. Case 2 uses the same `setup=` and `evals = 1`.

With both settings in place, the `result_recursive_memo::BenchmarkTools.Trial` variable holds the time for one memoized run that starts from an empty dictionary:

In [ ]:
result_recursive_memo = let
    test_run_recursive_memo = @benchmarkable memoization_fibonacci!($(benchmark_index), series) setup=(series = Dict{Int,Int}())
    result_recursive_memo = run(test_run_recursive_memo; samples = 200, evals = 1)
end

___

## Summary
Three implementations of one calculation, benchmarked at the same problem size, separate the cost of recursion from the cost of repeated work.

> __Key Takeaways:__
>
> * __The recursive Fibonacci function repeats calculations:__ The plain recursive function computes the same Fibonacci subproblems many times. These repeated calculations account for its longer runtime relative to the loop-based function.
> * __Memoization avoids repeated calculations:__ The memoized function checks its dictionary for a stored value before computing a subproblem. Each Fibonacci subproblem is therefore computed once and reused by later recursive calls.
> * __The benchmark setup affects the comparison:__ All three functions must use the same input value when their runtimes are compared. Each memoized benchmark sample must also begin with an empty dictionary so that the measured call includes the work of filling the cache.

For this calculation, memoization preserves the recursive implementation while removing repeated subproblem calculations.
___